In [1]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd, mne
from mne.time_frequency import psd_array_welch
from src.load import list_subjects, load_subject
from src.channels import CHANNELS, SFREQ

N = 4 * SFREQ                         # samples per 4 s segment = 512
segments, subject, label = [], [], []
for _, k in pd.DataFrame(list_subjects()).iterrows():
    x = mne.filter.filter_data(load_subject(k.path), SFREQ, 1.0, 40.0, verbose=False)
    x = x[:, :60 * SFREQ]                                       # keep the first 60 s only
    segments.append(x.reshape(19, 15, N).transpose(1, 0, 2))   # -> (15 segments, 19 ch, 512)
    subject += [k.subject_id] * 15
    label += [k.label] * 15

W = np.concatenate(segments)          # (1815, 19, 512)
subject, label = np.array(subject), np.array(label)
print(W.shape)

(1815, 19, 512)


In [2]:
psd, freqs = psd_array_welch(W, SFREQ, fmin=1, fmax=40, n_fft=N, verbose=False)
print(psd.shape)

def band(lo, hi):
    """Power between lo and hi Hz, for every segment and channel -> (1815, 19)."""
    return psd[..., (freqs >= lo) & (freqs < hi)].sum(axis=-1)

(1815, 19, 157)


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

def subject_level_auc(X, n_seeds=10):
    aucs = []
    for s in range(n_seeds):
        cv = StratifiedGroupKFold(5, shuffle=True, random_state=s)
        model = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=s)
        p = cross_val_predict(model, X, label, groups=subject, cv=cv, method="predict_proba")[:, 1]
        agg = pd.DataFrame({"subject": subject, "y": label, "p": p}).groupby("subject").mean()
        aucs.append(roc_auc_score(agg.y, agg.p))
    return np.mean(aucs), np.std(aucs)

In [4]:
theta, alpha, beta = band(4, 8), band(8, 13), band(13, 30)
total_1_40 = band(1, 41)
total_1_20 = band(1, 20)
emg        = band(20, 41)

edge = [CHANNELS.index(c) for c in ["T7", "T8", "F7", "F8"]]    # EMG-prone sites
rest = [i for i in range(19) if i not in edge]

A = np.hstack([theta/total_1_40, alpha/total_1_40, beta/total_1_40, theta/beta])   # all features
B = A.reshape(len(A), 4, 19)[:, :, rest].reshape(len(A), -1)                      # no edge channels
C = np.hstack([theta/total_1_20, alpha/total_1_20])                                # slow rhythms only
D = (emg / total_1_40)[:, edge]                                                    # EMG only

for name, X in zip("ABCD", [A, B, C, D]):
    print(name, X.shape)

A (1815, 76)
B (1815, 60)
C (1815, 38)
D (1815, 4)


In [5]:
for name, X in [("A all features", A), ("B no edge channels", B),
                ("C slow rhythms only", C), ("D EMG only", D)]:
    m, s = subject_level_auc(X)
    print(f"{name:22s} {X.shape[1]:3d} features   AUC {m:.2f} ± {s:.2f}")

A all features          76 features   AUC 0.69 ± 0.01
B no edge channels      60 features   AUC 0.71 ± 0.02
C slow rhythms only     38 features   AUC 0.69 ± 0.02
D EMG only               4 features   AUC 0.65 ± 0.02


In [6]:
front_and_edge = [CHANNELS.index(c) for c in ["Fp1", "Fp2", "F7", "F8", "T7", "T8"]]
inner = [i for i in range(19) if i not in front_and_edge]          # 13 inner channels

E = np.hstack([theta/total_1_20, alpha/total_1_20])[:, np.r_[inner, [i + 19 for i in inner]]]
F = (alpha / total_1_20)[:, inner]
G = (theta / total_1_20)[:, inner]

for name, X in [("E slow, no EOG/EMG sites", E), ("F alpha only, inner", F), ("G theta only, inner", G)]:
    m, s = subject_level_auc(X)
    print(f"{name:26s} {X.shape[1]:3d} features   AUC {m:.2f} ± {s:.2f}")

E slow, no EOG/EMG sites    26 features   AUC 0.71 ± 0.01
F alpha only, inner         13 features   AUC 0.69 ± 0.02
G theta only, inner         13 features   AUC 0.65 ± 0.03
